## 训练U-Net模型

新建python 3.10环境（以conda为例）

```
conda create -n hw4 python=3.10 -y
conda activate hw4
```

安装torch，注意cuda版本适配
```
pip install torch==2.0.* torchvision==0.15.* --index-url https://download.pytorch.org/whl/cu117
```

安装其他依赖库
```
pip install ipykernel==6.26.* matplotlib==3.8.* medpy==0.4.* scipy==1.11.* numpy==1.23.* scikit-image==0.22.* imageio==2.31.* tensorboard==2.15.* tqdm==4.* -i https://pypi.tuna.tsinghua.edu.cn/simple
```

In [ ]:
import json
import os

from types import SimpleNamespace
import tqdm
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from dataset import BrainSegmentationDataset as Dataset
from logger import Logger
from loss import DiceLoss
from transform import transforms
from unet import UNet
from utils import log_images, dsc

### 输入参数  
  
device: 设备编号  
batch_size: 批大小  
epochs: 训练轮数  
lr: 学习率  
vis_images: 可视化预测结果的数目 (在tensorboard中查看)  
vis_freq: 两次可视化预测结果的间隔  
weights: 训练后的模型参数路径    
images: 数据集路径   
image_size: 图像尺寸   
aug_scale: 数据增强(放缩)  
aug_angle: 数据增强(旋转)  

In [ ]:
args = SimpleNamespace(
    device="cuda:0",
    batch_size=16,
    epochs=100,
    lr=0.0001,
    workers=0,
    vis_images=200,
    vis_freq=10,
    weights="./weights",
    logs="./logs",
    images="./kaggle_3m",
    image_size=256,
    aug_scale=0.05,
    aug_angle=15,
    early_stop_patience=25,
    early_stop_min_delta=0.0001,
)

In [ ]:
# 读取数据
def worker_init(worker_id):
    # ============【修改点1】============
    # 修改随机数种子
    np.random.seed(2025 + worker_id)
    # ==================================


def data_loaders(args):
    dataset_train, dataset_valid = datasets(args)

    loader_train = DataLoader(
        dataset_train,
        batch_size=args.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=args.workers,
        worker_init_fn=worker_init,
    )
    loader_valid = DataLoader(
        dataset_valid,
        batch_size=args.batch_size,
        drop_last=False,
        num_workers=args.workers,
        worker_init_fn=worker_init,
    )

    return dataset_train, dataset_valid, loader_train, loader_valid


# 数据集定义
def datasets(args):
    train = Dataset(
        images_dir=args.images,
        subset="train",
        image_size=args.image_size,
        transform=transforms(scale=args.aug_scale, angle=args.aug_angle, flip_prob=0.5),
    )
    valid = Dataset(
        images_dir=args.images,
        subset="validation",
        image_size=args.image_size,
        random_sampling=False,
    )
    return train, valid


# 数据处理
# ============【修改点2】============
# 修改dsc_per_volume函数，添加异常处理
def dsc_per_volume(validation_pred, validation_true, patient_slice_index):
    """
    计算每个volume的DSC分数
    添加异常处理以应对空预测的情况
    """
    dsc_list = []
    num_slices = np.bincount([p[0] for p in patient_slice_index])
    index = 0
    for p in range(len(num_slices)):
        y_pred = np.array(validation_pred[index : index + num_slices[p]])
        y_true = np.array(validation_true[index : index + num_slices[p]])
        try:
            dsc_score = dsc(y_pred, y_true)
            dsc_list.append(dsc_score)
        except (ValueError, IndexError) as e:
            # 如果预测结果为空或出现其他问题，使用0作为DSC分数
            print(f"Warning: DSC calculation failed for volume {p}: {str(e)}")
            dsc_list.append(0.0)
        index += num_slices[p]
    return dsc_list
# ===================================


def log_loss_summary(logger, loss, step, prefix=""):
    logger.scalar_summary(prefix + "loss", np.mean(loss), step)


def makedirs(args):
    os.makedirs(args.weights, exist_ok=True)
    os.makedirs(args.logs, exist_ok=True)


def snapshotargs(args):
    args_file = os.path.join(args.logs, "args.json")
    with open(args_file, "w") as fp:
        json.dump(vars(args), fp)

In [ ]:
class EarlyStopping:
    """
    早停机制类
    """

    def __init__(self, patience=25, min_delta=0.0001, mode="max"):
        """
        Args:
            patience: 容忍的epoch数，在此期间没有改进则停止
            min_delta: 最小改进阈值，小于此值不算改进
            mode: 'max'表示指标越大越好(如DSC)，'min'表示越小越好(如loss)
        """
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, score):
        """
        检查是否需要早停
        Args:
            score: 当前的评估指标
        Returns:
            bool: 是否应该早停
        """

        if self.best_score is None:
            self.best_score = score
            return False

        # 根据mode判断是否有改进
        if self.mode == "max":
            improved = score > self.best_score + self.min_delta
        else:
            improved = score < self.best_score - self.min_delta

        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                return True

        return False

In [ ]:
makedirs(args)
snapshotargs(args)
device = torch.device("cpu" if not torch.cuda.is_available() else args.device)

dataset_train, dataset_valid, loader_train, loader_valid = data_loaders(args)
loaders = {"train": loader_train, "valid": loader_valid}

In [ ]:
unet = UNet(in_channels=Dataset.in_channels, out_channels=Dataset.out_channels)
unet.to(device)

dsc_loss = DiceLoss()
best_validation_dsc = 0.0

optimizer = optim.AdamW(unet.parameters(), lr=args.lr)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=args.lr * 10,
    epochs=args.epochs,
    steps_per_epoch=len(loader_train),
    pct_start=0.3,
    anneal_strategy="cos",
    div_factor=25.0,
    final_div_factor=10000.0,
)

early_stopping = EarlyStopping(
    patience=args.early_stop_patience,
    min_delta=args.early_stop_min_delta,
    mode="max",
)

logger = Logger(args.logs)
loss_train = []
loss_valid = []

step = 0

In [ ]:
for epoch in range(args.epochs):
    for phase in ["train", "valid"]:
        if phase == "train":
            unet.train()
        else:
            unet.eval()

        validation_pred = []
        validation_true = []

        for i, data in enumerate(tqdm.tqdm(loaders[phase])):
            if phase == "train":
                step += 1

            x, y_true = data
            x, y_true = x.to(device), y_true.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(phase == "train"):
                y_pred = unet(x)

                loss = dsc_loss(y_pred, y_true)

                if phase == "valid":
                    loss_valid.append(loss.item())
                    y_pred_np = y_pred.detach().cpu().numpy()
                    validation_pred.extend(
                        [y_pred_np[s] for s in range(y_pred_np.shape[0])]
                    )
                    y_true_np = y_true.detach().cpu().numpy()
                    validation_true.extend(
                        [y_true_np[s] for s in range(y_true_np.shape[0])]
                    )
                    if (epoch % args.vis_freq == 0) or (epoch == args.epochs - 1):
                        if i * args.batch_size < args.vis_images:
                            tag = "image/{}".format(i)
                            num_images = args.vis_images - i * args.batch_size
                            logger.image_list_summary(
                                tag,
                                log_images(x, y_true, y_pred)[:num_images],
                                step,
                            )

                if phase == "train":
                    loss_train.append(loss.item())
                    loss.backward()
                    optimizer.step()
                    scheduler.step()

            if phase == "train" and (step + 1) % 10 == 0:
                log_loss_summary(logger, loss_train, step)
                loss_train = []

        if phase == "valid":
            log_loss_summary(logger, loss_valid, step, prefix="val_")
            print("epoch {} | val_loss: {}".format(epoch + 1, np.mean(loss_valid)))
            mean_dsc = np.mean(
                dsc_per_volume(
                    validation_pred,
                    validation_true,
                    loader_valid.dataset.patient_slice_index,
                )
            )
            logger.scalar_summary("val_dsc", mean_dsc, step)
            print("epoch {} | val_dsc: {}".format(epoch + 1, mean_dsc))
            if mean_dsc > best_validation_dsc:
                best_validation_dsc = mean_dsc
                torch.save(unet.state_dict(), os.path.join(args.weights, "unet.pt"))
            loss_valid = []
            if early_stopping(mean_dsc):
                print(f"Early stopping triggered at epoch {epoch + 1}")
                print(f"Best validation DSC: {best_validation_dsc:.4f}")
                break

    if early_stopping.early_stop:
        break

    current_lr = optimizer.param_groups[0]["lr"]
    print("epoch {} | learning rate: {:.6f}".format(epoch + 1, current_lr))

print("Best validation mean DSC: {:4f}".format(best_validation_dsc))